In [ ]:
%pip install -q ultralytics opencv-contrib-python transformers datasets accelerate scikit-learn pandas matplotlib pillow

In [ ]:
import os, time, math, json, random
from pathlib import Path
from collections import Counter, defaultdict, deque

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

from ultralytics import YOLO
from transformers import AutoImageProcessor, AutoModelForImageClassification

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("Device:", TORCH_DEVICE)

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"
for folder in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
CONFIDENCE_THRESHOLD = 0.35
TRACKER_CONFIG = "bytetrack.yaml"
MAX_TRAJECTORY_LENGTH = 40
LINE_CROSSING_ENABLED = True
LINE_X_RATIO = 0.50
SPEED_THRESHOLD_PX_S = 20.0

In [ ]:
BASE_YOLO_WEIGHTS = "yolo11n.pt"
model = YOLO(BASE_YOLO_WEIGHTS)
print("Loaded:", BASE_YOLO_WEIGHTS)
print("Classes:", len(model.names))

In [ ]:
CUSTOM_DATA_YAML = DATA_DIR / "my_dataset" / "data.yaml"
RUN_CUSTOM_YOLO_TRAINING = False  

In [ ]:
if RUN_CUSTOM_YOLO_TRAINING:
    if not CUSTOM_DATA_YAML.exists():
        raise FileNotFoundError(
            f"Dataset YAML not found: {CUSTOM_DATA_YAML}. "
            "Create the dataset structure and data.yaml described above."
        )

    base_model = YOLO(BASE_YOLO_WEIGHTS)
    train_results = base_model.train(
        data=str(CUSTOM_DATA_YAML),
        epochs=50,
        imgsz=640,
        batch=16,
        patience=10,
        device=DEVICE,
        workers=2,
        project=str(OUTPUT_DIR / "yolo_training"),
        name="custom_yolo11n",
        pretrained=True,
        plots=True,
        seed=SEED
    )

    best_weights = Path(base_model.trainer.best)
    if not best_weights.exists():
        raise FileNotFoundError("Training finished but best.pt was not found.")
    model = YOLO(str(best_weights))
    CUSTOM_YOLO_WEIGHTS = best_weights
    print("Custom model saved at:", best_weights)
else:
    print("Custom YOLO training skipped. Set RUN_CUSTOM_YOLO_TRAINING=True when your dataset is ready.")

In [ ]:
if CUSTOM_DATA_YAML.exists():
    val_metrics = model.val(data=str(CUSTOM_DATA_YAML), imgsz=640, device=DEVICE)
    print("mAP50:", float(val_metrics.box.map50))
    print("mAP50-95:", float(val_metrics.box.map))
    print("Precision (mean):", float(val_metrics.box.mp))
    print("Recall (mean):", float(val_metrics.box.mr))
else:
    print("No custom dataset YAML found. Add your dataset to calculate custom validation metrics.")

In [ ]:
SAMPLE_IMAGE = "https://ultralytics.com/images/bus.jpg"
image_results = model.predict(SAMPLE_IMAGE, conf=CONFIDENCE_THRESHOLD, verbose=False)
display(Image.fromarray(cv2.cvtColor(image_results[0].plot(), cv2.COLOR_BGR2RGB)))
if image_results[0].boxes is not None and len(image_results[0].boxes):
    confs = image_results[0].boxes.conf.cpu().numpy()
    print("Detections:", len(confs))
    print("Mean detection confidence:", round(float(confs.mean()), 4))
else:
    print("No objects detected.")

In [ ]:
EXPRESSION_MODEL_NAME = "trpakov/vit-face-expression"
expression_processor = AutoImageProcessor.from_pretrained(EXPRESSION_MODEL_NAME)
expression_model = AutoModelForImageClassification.from_pretrained(EXPRESSION_MODEL_NAME)
expression_model = expression_model.to(TORCH_DEVICE).eval()

In [ ]:
id2label = expression_model.config.id2label
print("Expression labels:", id2label)

In [ ]:
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_detector = cv2.CascadeClassifier(cascade_path)
if face_detector.empty():
    raise RuntimeError(
        "Haar Cascade XML could not be loaded. Check your OpenCV installation "
        "and restart the kernel after package changes."
    )
print("Face detector ready.")

In [ ]:
def predict_expression(face_bgr):
    if face_bgr is None or face_bgr.size == 0:
        return "Unknown", 0.0

    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    pil_face = Image.fromarray(face_rgb)
    inputs = expression_processor(images=pil_face, return_tensors="pt")
    inputs = {k: v.to(TORCH_DEVICE) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = expression_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]

    idx = int(torch.argmax(probs).item())
    return id2label[idx], float(probs[idx].item() * 100)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from transformers import get_linear_schedule_with_warmup

In [ ]:
EXPRESSION_DATA_DIR = DATA_DIR / "expressions"
RUN_EXPRESSION_FINE_TUNING = False  

In [ ]:
class ExpressionImageDataset(Dataset):
    def __init__(self, root, processor, class_to_idx=None):
        self.dataset = ImageFolder(root)
        self.processor = processor
        if class_to_idx is not None and self.dataset.class_to_idx != class_to_idx:
            raise ValueError("Train and validation class folders must match.")
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        inputs = self.processor(images=image.convert("RGB"), return_tensors="pt")
        return {k: v.squeeze(0) for k, v in inputs.items()} | {"labels": torch.tensor(label)}

In [ ]:
def collate_expression(batch):
    keys = [k for k in batch[0] if k != "labels"]
    result = {k: torch.stack([item[k] for item in batch]) for k in keys}
    result["labels"] = torch.stack([item["labels"] for item in batch])
    return result

if RUN_EXPRESSION_FINE_TUNING:
    train_dir = EXPRESSION_DATA_DIR / "train"
    val_dir = EXPRESSION_DATA_DIR / "val"
    if not train_dir.exists() or not val_dir.exists():
        raise FileNotFoundError("Create data/expressions/train and data/expressions/val class folders first.")

    train_ds = ExpressionImageDataset(train_dir, expression_processor)
    val_ds = ExpressionImageDataset(val_dir, expression_processor, train_ds.dataset.class_to_idx)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_expression)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_expression)

    custom_labels = {v: k for k, v in train_ds.dataset.class_to_idx.items()}
    if expression_model.config.num_labels != len(custom_labels):
        expression_model = AutoModelForImageClassification.from_pretrained(
            EXPRESSION_MODEL_NAME,
            num_labels=len(custom_labels),
            ignore_mismatched_sizes=True
        )
    if hasattr(expression_model, "classifier"):
        torch.nn.init.xavier_uniform_(expression_model.classifier.weight)
        if expression_model.classifier.bias is not None:
            torch.nn.init.zeros_(expression_model.classifier.bias)
    expression_model.config.id2label = custom_labels
    expression_model.config.label2id = train_ds.dataset.class_to_idx
    expression_model.to(TORCH_DEVICE)

    optimizer = torch.optim.AdamW(expression_model.parameters(), lr=2e-5)
    epochs = 3
    total_steps = epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    for epoch in range(epochs):
        expression_model.train()
        train_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(TORCH_DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = expression_model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(expression_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += float(out.loss.item())

        expression_model.eval()
        correct = total = 0
        with torch.inference_mode():
            for batch in val_loader:
                batch = {k: v.to(TORCH_DEVICE) for k, v in batch.items()}
                logits = expression_model(**{k: v for k, v in batch.items() if k != "labels"}).logits
                preds = logits.argmax(dim=1)
                correct += int((preds == batch["labels"]).sum().item())
                total += int(batch["labels"].numel())

        print(f"Epoch {epoch+1}/{epochs} | loss={train_loss/max(1,len(train_loader)):.4f} | val_accuracy={correct/max(1,total):.4f}")

    CUSTOM_EXPRESSION_DIR = MODEL_DIR / "custom_expression_vit"
    expression_model.save_pretrained(CUSTOM_EXPRESSION_DIR)
    expression_processor.save_pretrained(CUSTOM_EXPRESSION_DIR)
    print("Fine-tuned expression model saved to:", CUSTOM_EXPRESSION_DIR)
else:
    print("Expression fine-tuning skipped. Set RUN_EXPRESSION_FINE_TUNING=True when your labelled dataset is ready.")

In [ ]:
def run_expression_webcam(camera_index=0, frame_skip=5):
    cap = cv2.VideoCapture(camera_index)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
    if not cap.isOpened():
        raise RuntimeError("Unable to access webcam.")

    frame_no = 0
    cached = {}
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_no += 1
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_detector.detectMultiScale(
                gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
            )
            updated = {}
            for i, (x, y, w, h) in enumerate(faces):
                crop = frame[y:y+h, x:x+w]
                if frame_no % frame_skip == 0 or i not in cached:
                    cached[i] = predict_expression(crop)
                label, confidence = cached[i]
                updated[i] = cached[i]
                cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
                cv2.putText(frame, f"{label} {confidence:.1f}%", (x,max(25,y-10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            cached = updated
            cv2.imshow("Facial Expression Recognition - Q to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
def run_integrated_system(camera_index=0, frame_skip=5):
    cap = cv2.VideoCapture(camera_index)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
    if not cap.isOpened():
        raise RuntimeError("Unable to access webcam.")

    history = {}
    unique_ids = set()
    left_to_right = 0
    right_to_left = 0
    frame_no = 0
    total_fps = 0.0
    expression_cache = {}
    expression_summary = Counter()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_no += 1
            start = time.perf_counter()
            h, w = frame.shape[:2]
            line_x = int(w * LINE_X_RATIO)

            result = model.track(
                source=frame, persist=True, tracker=TRACKER_CONFIG,
                conf=CONFIDENCE_THRESHOLD, device=DEVICE, verbose=False
            )[0]
            annotated = result.plot()
            current_counts = Counter()
            current_speeds = []

            if LINE_CROSSING_ENABLED:
                cv2.line(annotated, (line_x, 0), (line_x, h), (255,0,255), 2)

            if result.boxes is not None and result.boxes.id is not None:
                ids = result.boxes.id.int().cpu().tolist()
                classes = result.boxes.cls.int().cpu().tolist()
                boxes = result.boxes.xyxy.cpu().numpy()
                now = time.perf_counter()

                for track_id, class_id, box in zip(ids, classes, boxes):
                    x1,y1,x2,y2 = map(int, box)
                    cx,cy = (x1+x2)//2, (y1+y2)//2
                    class_name = model.names[class_id]
                    current_counts[class_name] += 1
                    unique_ids.add(track_id)

                    if track_id not in history:
                        history[track_id] = {
                            "points": deque(maxlen=MAX_TRAJECTORY_LENGTH),
                            "prev_center": None, "prev_time": None,
                            "prev_side": None
                        }
                    item = history[track_id]
                    item["points"].append((cx,cy))

                    speed = 0.0
                    if item["prev_center"] is not None and item["prev_time"] is not None:
                        dt = now - item["prev_time"]
                        if dt > 0:
                            dx,dy = cx-item["prev_center"][0], cy-item["prev_center"][1]
                            speed = math.hypot(dx,dy)/dt
                    item["prev_center"], item["prev_time"] = (cx,cy), now
                    current_speeds.append(speed)

                    side = "left" if cx < line_x-5 else ("right" if cx > line_x+5 else item["prev_side"])
                    if LINE_CROSSING_ENABLED and side and item["prev_side"] and side != item["prev_side"]:
                        if item["prev_side"] == "left" and side == "right":
                            left_to_right += 1
                        elif item["prev_side"] == "right" and side == "left":
                            right_to_left += 1
                    if side:
                        item["prev_side"] = side

                    pts = list(item["points"])
                    for j in range(1, len(pts)):
                        cv2.line(annotated, pts[j-1], pts[j], (0,255,255), 2)
                    cv2.putText(annotated, f"ID {track_id} | {speed:.0f}px/s",
                                (x1, max(20,y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                                (0,255,255), 1)

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_detector.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
            current_face_cache = {}
            for face_i, (x,y,fw,fh) in enumerate(faces):
                crop = frame[y:y+fh, x:x+fw]
                if frame_no % frame_skip == 0 or face_i not in expression_cache:
                    expression_cache[face_i] = predict_expression(crop)
                label, score = expression_cache[face_i]
                current_face_cache[face_i] = (label, score)
                expression_summary[label] += 1
                cv2.rectangle(annotated, (x,y), (x+fw,y+fh), (0,200,255), 2)
                cv2.putText(annotated, f"Face: {label} {score:.0f}%",
                            (x,max(20,y-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0,200,255), 2)
            expression_cache = current_face_cache

            elapsed = time.perf_counter() - start
            fps = 1.0/elapsed if elapsed > 0 else 0.0
            total_fps += fps
            avg_fps = total_fps/frame_no
            panel = [
                f"Objects visible: {sum(current_counts.values())}",
                f"Unique tracked IDs: {len(unique_ids)}",
                f"Left -> Right: {left_to_right}",
                f"Right -> Left: {right_to_left}",
                f"Mean speed: {np.mean(current_speeds) if current_speeds else 0:.1f} px/s",
                f"FPS: {fps:.1f} | Avg: {avg_fps:.1f}",
                f"Faces detected: {len(faces)}"
            ]
            y0 = 25
            for line in panel:
                cv2.putText(annotated, line, (10,y0), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (50,255,50), 1)
                y0 += 22

            cv2.imshow("Integrated Detection, Tracking & Expression Analytics", annotated)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

    print("Frames processed:", frame_no)
    print("Unique tracked object IDs:", len(unique_ids))
    print("Left-to-right crossings:", left_to_right)
    print("Right-to-left crossings:", right_to_left)
    print("Average FPS:", round(total_fps/frame_no, 2) if frame_no else 0)
    print("Expression prediction counts:", dict(expression_summary))

In [ ]:
YOLO_EXPORT_PATH = MODEL_DIR / ("custom_yolo11n.pt" if "CUSTOM_YOLO_WEIGHTS" in globals() else "pretrained_yolo11n.pt")
model.save(str(YOLO_EXPORT_PATH))
EXPRESSION_EXPORT_DIR = MODEL_DIR / "expression_vit"
expression_model.save_pretrained(EXPRESSION_EXPORT_DIR)
expression_processor.save_pretrained(EXPRESSION_EXPORT_DIR)
print("YOLO weights:", YOLO_EXPORT_PATH)
print("Expression model:", EXPRESSION_EXPORT_DIR)

In [ ]:
if RUN_EXPRESSION_FINE_TUNING and "val_loader" in globals():
    from sklearn.metrics import classification_report, confusion_matrix
    expression_model.eval()
    y_true, y_pred = [], []
    with torch.inference_mode():
        for batch in val_loader:
            labels = batch["labels"].numpy().tolist()
            inputs = {k: v.to(TORCH_DEVICE) for k,v in batch.items() if k != "labels"}
            preds = expression_model(**inputs).logits.argmax(dim=1).cpu().numpy().tolist()
            y_true.extend(labels)
            y_pred.extend(preds)
    names = [custom_labels[i] for i in range(len(custom_labels))]
    print(classification_report(y_true, y_pred, target_names=names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    plt.imshow(cm)
    plt.xticks(range(len(names)), names, rotation=45, ha="right")
    plt.yticks(range(len(names)), names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Facial Expression Validation Confusion Matrix")
    plt.colorbar()
    plt.tight_layout()
    plt.show()
else:
    print("Run expression fine-tuning first to calculate held-out classification metrics.")